# Student Dropout Risk & Academic Performance

Leakage-safe notebook for the IRA project. Raw CSV is never overwritten.

**Classification:** dropout risk — Low / Medium / High  
**Regression:** academic performance — mean of current test and assignment scores

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
RAW_CSV = Path("synthetic_student_data_with_risk_30_40_30.csv")
assert RAW_CSV.exists(), "Raw dataset missing" 

## 1. Load raw data (preserved)

In [ ]:
df_raw = pd.read_csv(RAW_CSV)
print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.head()

In [ ]:
df_raw.info()
print("\nMissing values:\n", df_raw.isnull().sum())
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("\nTarget:\n", df_raw["dropout_target"].value_counts(normalize=True).round(4))
print("\nFees:\n", df_raw["fees"].value_counts())
print("\nGender:\n", df_raw["gender"].value_counts())
df_raw.describe()

## 2. Cleaning (working copy only)

- Drop 19 exact duplicates
- No missing values; imputers still live in the inference pipeline
- Score/attendance ranges 0–100 are valid; RobustScaler is used instead of deleting outliers
- Mild class mix (~30/40/30): `class_weight='balanced'`
- **No scaling or encoding before the train/test split** (that leaked in the original notebook)

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
df["academic_performance"] = (df["current_test_score"] + df["current_assignment_score"]) / 2.0
print("Clean rows:", len(df), "removed", len(df_raw) - len(df), "duplicates")

## 3. EDA

In [ ]:
order = ["Low Risk", "Medium Risk", "High Risk"]
colors = {"High Risk": "#e74c3c", "Medium Risk": "#f1c40f", "Low Risk": "#2ecc71"}
counts = df["dropout_target"].value_counts().reindex(order)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].barh(counts.index, counts.values, color=[colors[i] for i in counts.index])
axes[0].set_title("Dropout risk")
sns.countplot(data=df, x="fees", ax=axes[1], hue="fees", legend=False)
axes[1].set_title("Fees")
sns.countplot(data=df, x="gender", ax=axes[2], hue="gender", legend=False)
axes[2].set_title("Gender")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = [
    "attendance", "current_test_score", "current_assignment_score",
    "previous_test_score", "previous_assignment_score", "academic_performance",
]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], bins=30, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
corr_df = df[num_cols].corr()
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Numeric correlations (no target encoding)")
plt.tight_layout()
plt.show()

## 4. Train models (reproducible)

Training lives in `ml/train.py`: ColumnTransformer + Pipeline, 80/20 split (`random_state=42`, stratified for classification).

Classification candidates: Logistic Regression, Decision Tree, Random Forest, HistGradientBoosting (best by **F1-macro**).  
Regression candidates: Linear, Ridge, Decision Tree, Random Forest, HistGradientBoosting (best by **R²**).

In [ ]:
import sys
sys.path.insert(0, str(Path("ml").resolve().parent))
from ml.train import main as train_main
train_main()

## 5. Report saved metrics (computed, not invented)

In [ ]:
info = json.loads(Path("ml/artifacts/model_info.json").read_text())
print("Best classifier:", info["classification"]["best_model"])
print(json.dumps(info["classification"]["test_metrics"], indent=2))
print("\nPer-class:")
print(json.dumps(info["classification"]["per_class"], indent=2))
print("\nBest regressor:", info["regression"]["best_model"])
print(json.dumps(info["regression"]["test_metrics"], indent=2))
print("\nCandidates (classification):")
print(pd.DataFrame(info["classification"]["candidates"]))
print("\nCandidates (regression):")
print(pd.DataFrame(info["regression"]["candidates"]))

In [ ]:
labels = info["classification"]["confusion_matrix_labels"]
cm = np.array(info["classification"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

## 6. Inference artifact

Pipelines written to `ml/artifacts/dropout_pipeline.joblib` and `performance_pipeline.joblib`.  
The Node API calls `ml/infer.py` — it does **not** retrain.

In [ ]:
import joblib
clf = joblib.load("ml/artifacts/dropout_pipeline.joblib")
reg = joblib.load("ml/artifacts/performance_pipeline.joblib")
sample = pd.DataFrame([{
    "attendance": 49,
    "current_test_score": 46,
    "current_assignment_score": 50,
    "previous_test_score": 29,
    "previous_assignment_score": 28,
    "fees": "Paid",
    "gender": "Male",
}])
print("Risk:", clf.predict(sample)[0])
print("Proba:", dict(zip(clf.classes_, clf.predict_proba(sample)[0].round(4))))
print("Predicted performance:", round(float(reg.predict(sample[["attendance","previous_test_score","previous_assignment_score","fees","gender"]])[0]), 2))